# Stage 3.B — distributed aero-first BO campaign (RE-RUN, coordination fixed)

**Why a re-run:** the first attempt fragmented — Google Drive made duplicate `blade_campaign` folders when the 3 sessions raced to create it, so they never shared one ledger. This version **fixes that**: cell 4 (PREP) creates ONE fresh folder (`campaign_run2`) and seeds it with ALL designs from the broken run, and cell 5 (GUARD) refuses to start unless exactly one shared folder exists. So we **continue** from the prior work toward 300 designs of *pooled* learning — nothing is wasted.

**How to launch (order matters):** run **Session 0 first** through the RUN cell (its PREP seeds the folder). Once it's going, run **Sessions 1 & 2** — their GUARD waits for Session 0's folder to sync, then they join. Set only `SESSION_INDEX` (0/1/2) per session.

**Async, verified:** a seconds-long pre-flight (cell 6) smoke-tests the async loop before the real run, and the RUN cell streams a LIVE per-completion async verdict. Resumable: re-running RUN after a Colab drop continues from the ledger.

## 1. Repo + deps + SU2 + Drive

In [ ]:
import importlib.util, os, subprocess, sys
from pathlib import Path
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ.setdefault(_v, "1")   # 1 thread/worker -> N processes on N cores

IN_COLAB = importlib.util.find_spec("google.colab") is not None
BRANCH = "main"  # the Stage-3 campaign machinery is merged to main
REPO = Path("/content/fan-optimization") if IN_COLAB else Path.cwd()
if IN_COLAB:
    if not REPO.exists():
        subprocess.run(["git", "clone", "-b", BRANCH,
                        "https://github.com/clingergab/fan-optimization.git", str(REPO)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO), "fetch", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO), "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO), "pull", "origin", BRANCH], check=True)
    subprocess.run("apt-get install -qq -y libglu1-mesa libxrender1 libxcursor1 "
                   "libxft2 libxinerama1 unzip".split(), check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}[bo]"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gmsh", "cadquery"], check=True)
    from google.colab import drive; drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/fanopt")
else:
    DRIVE_ROOT = REPO / "data"
for p in (str(REPO), str(REPO / "src"), str(REPO / "scripts")):
    if p not in sys.path:
        sys.path.insert(0, p)
print("repo:", REPO, "| drive:", DRIVE_ROOT)

In [ ]:
import urllib.request
from fanopt.cfd.phase3 import find_su2
SU2_BIN = find_su2()
if SU2_BIN is None and IN_COLAB:
    LOCAL = Path("/content/su2")
    if not any(LOCAL.rglob("SU2_CFD")):
        zc = DRIVE_ROOT / "su2" / "SU2-v8.0.1-linux64.zip"
        if not zc.exists():
            zc.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(
                "https://github.com/su2code/SU2/releases/download/v8.0.1/SU2-v8.0.1-linux64.zip", str(zc))
        LOCAL.mkdir(parents=True, exist_ok=True)
        subprocess.run(["unzip", "-q", "-o", str(zc), "-d", str(LOCAL)], check=True)
    hit = next(LOCAL.rglob("SU2_CFD"), None)
    if hit: subprocess.run(["chmod", "+x", str(hit)], check=False)
    SU2_BIN = str(hit) if hit else None
assert SU2_BIN, "SU2 not found"
print("SU2:", SU2_BIN)

## 2. Campaign config (edit `SESSION_INDEX` per session)

In [ ]:
# ---- EDIT PER SESSION -------------------------------------------------------------------
SESSION_INDEX = 0        # 0 in the FIRST session, 1 in the second, 2 in the third
N_SESSIONS    = 3        # how many Colab sessions you are running in total
SESSION_ID    = f"colab-{SESSION_INDEX}"
# ---- SHARED (identical in every session) ------------------------------------------------
N_MORE     = 100         # run this many ADDITIONAL designs beyond what's already pooled (seed + any
# prior run). The run cell sets budget = current_pooled_count + N_MORE, so it reliably adds N_MORE
# *new* progressive designs no matter how big the seed is (a fixed BUDGET=300 would add almost none).
N_INIT     = 24          # cold-start Sobol DoE (skipped automatically since the seed already exceeds it)
N_WORKERS  = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else (os.cpu_count() or 1)
BATCH_SIZE = N_WORKERS   # match the core count so no worker sits idle (q per session; 3x = total in-flight)
EXPLORE_FRACTION = 0.25  # BO + exploration: 1/4 of BO-phase dispatches are space-filling Sobol
# draws (not acquisition-driven), so the search keeps probing new regions — a hedge against the
# fast convergence we saw. Set 0.0 for pure exploit.
# Fidelity — LOCKED coarse tier from the Stage-2 (2a) probe: coarse preserves the fine ranking
# (Kendall tau=1.0), so the campaign explores on coarse; fine (5,60) is reserved for the 3.C
# top-cluster confirmation. See ADR-0004.
N_CYCLES   = 3           # coarse campaign tier
INNER_ITER = 30
# FRESH folder for the fixed re-run — a unique name that does NOT collide with the corrupted
# blade_campaign* folders from the first attempt. Prep (cell 4) seeds it with all prior designs.
SHARED_DIR = DRIVE_ROOT / "campaign_run2"
print(f"session {SESSION_ID} of {N_SESSIONS} | {N_WORKERS} workers | +{N_MORE} more | shared {SHARED_DIR}")

## 3. PREP — seed the fresh shared folder from the first run (Session 0 only)

In [ ]:
# ============================================================================================
# PREP — creates the shared folder ONCE and seeds it with ALL designs from the first (broken) run,
# so we CONTINUE (not restart). Only Session 0 does this; Sessions 1 & 2 skip it and wait in cell 5.
# Safe to Run-All in every session (non-0 sessions no-op here). Idempotent (re-runs skip).
# ============================================================================================
import json
from fanopt.bo.campaign_analysis import find_shards, load_rows
if SESSION_INDEX == 0:
    SHARED_DIR.mkdir(parents=True, exist_ok=True)
    seed = SHARED_DIR / "evaluations_seed.jsonl"
    if seed.exists():
        print("seed already present — skipping consolidation")
    else:
        old = [s for shs in find_shards(DRIVE_ROOT, "blade_campaign*").values() for s in shs]
        rows = load_rows(old)
        uniq = {}
        for r in rows:
            uniq.setdefault(r.get("design_hash"), r)
        # strip the first run's per-eval TELEMETRY (dispatch/finish times use that run's clock) so
        # the new run's live async check isn't poisoned; keep design vector + objectives for the GP.
        drop = ("dispatch_s", "finish_s", "inflight_at_dispatch")
        clean = [{k: v for k, v in r.items() if k not in drop} for r in uniq.values()]
        seed.write_text("\n".join(json.dumps(r) for r in clean) + "\n")
        print(f"seeded {len(clean)} unique designs from {len(old)} old shards -> {seed}")
else:
    print(f"Session {SESSION_INDEX}: PREP is Session-0 only — skipping (cell 5 will wait for the folder).")

## 4. Safety guard — one shared folder, no duplicates (every session)

In [ ]:
# ============================================================================================
# SAFETY GUARD (run in EVERY session) — waits for the ONE shared folder, refuses to start on
# duplicates, verifies the seed is codec-compatible, and locks this SESSION_INDEX. This is the fix.
# ============================================================================================
import json, time
from fanopt.bo.blade_codec import N_DIMS
seed = SHARED_DIR / "evaluations_seed.jsonl"
for _ in range(60):  # wait up to 10 min for Session 0's folder+seed to sync to this VM
    if seed.exists():
        break
    print("waiting for the shared folder + seed from Session 0 ...", flush=True); time.sleep(10)
assert seed.exists(), "no shared seed — run PREP in Session 0 first."
dups = sorted(p for p in DRIVE_ROOT.glob(SHARED_DIR.name + "*") if p.is_dir())
assert len(dups) == 1, f"DUPLICATE shared folders exist: {dups}\nDelete the extras in Drive before launching."
# the seed must match the CURRENT codec, else read_ledger silently drops every seed row (restart,
# not continue). Catch it now instead of after burning compute.
_v0 = json.loads(seed.read_text().splitlines()[0])["vector"]
assert len(_v0) == N_DIMS, (f"seed vectors are {len(_v0)}-D but the codec is {N_DIMS}-D — the first "
    f"run used a DIFFERENT codec, so the seed is INCOMPATIBLE and would be silently discarded. STOP.")
# lock this SESSION_INDEX so two sessions can't accidentally share an id/shard (footgun that could
# re-trigger the original bug). A fresh lock (<5 min) means another live session already took it.
_lock = SHARED_DIR / f"session_{SESSION_INDEX}.lock"
try:
    with open(_lock, "x") as f: f.write(SESSION_ID)
except FileExistsError:
    if time.time() - _lock.stat().st_mtime < 300:
        raise AssertionError(f"SESSION_INDEX={SESSION_INDEX} is already in use by a session started "
                             f"<5 min ago — give THIS session a different SESSION_INDEX (0/1/2).")
    _lock.write_text(SESSION_ID)  # stale lock (a resume) — take it over
print(f"guard OK: one shared folder, seed is {len(_v0)}-D (matches codec), SESSION_INDEX {SESSION_INDEX} locked.")

## 5. Async pre-flight — seconds, no CFD; if it fails, do not launch

In [ ]:
from fanopt.bo.distributed_campaign import preflight_async_check
# Smoke test with a FAST dummy objective (NO CFD) — confirms the async MACHINERY works in THIS Colab
# runtime BEFORE committing to the multi-hour campaign: the pool fills to N_WORKERS and REFILLS on
# completion (reaches 2*N_WORKERS unique, no duplicates). Takes ~1 min at 12 workers.
pf = preflight_async_check(n_workers=N_WORKERS)
ps = pf["per_session"]["preflight"]
print(f"async pre-flight (no CFD): peak_concurrency={ps['peak_concurrency']}/{N_WORKERS}  "
      f"refilled={pf['reached_budget']}  no_duplicates={pf['no_duplicates']}  passed={pf['passed']}")
print(f"  (smoke utilization={ps['utilization']:.0%} — LOW is EXPECTED here: the fast dummy objective "
      f"makes the serial GP proposal the bottleneck. Real async utilization is measured LIVE in the "
      f"run cell, where 2.8h evals dwarf the proposal.)")
assert pf["passed"], "ASYNC PRE-FLIGHT FAILED — pool didn't fill/refill on completion; do NOT launch."
print("OK — async dispatch-on-completion works in this runtime. Launch below; watch the run cell's "
      "live utilization verdict on the real objective.")

## 6. Run the session — streams a LIVE async verdict per completion (resumable)

In [ ]:
import time
import run_blade_campaign_distributed as campaign
import fanopt.geometry.blade_cad as blade_cad
from fanopt.bo.distributed_campaign import read_ledger
blade_cad.N_RADIAL_SECTIONS = 40  # the objective's geometry resolution (ADR-0004)

CLAIM_TTL = 6 * 3600  # must exceed the per-eval wall time — measured ~3.6-3.85h, so 6h with margin
# budget = (what's already pooled RIGHT NOW: seed + any prior evals) + N_MORE, so this session drives
# toward N_MORE additional designs regardless of seed size (see N_MORE note in the config cell).
_n0 = len(read_ledger(SHARED_DIR)[0]); _t0 = time.time()
BUDGET = _n0 + N_MORE
print(f"pooled so far: {_n0} designs -> target budget {BUDGET} (+{N_MORE} more)")
argv = ["--shared-dir", str(SHARED_DIR), "--session-id", SESSION_ID,
        "--session-index", str(SESSION_INDEX), "--n-sessions", str(N_SESSIONS),
        "--budget", str(BUDGET), "--n-init", str(N_INIT), "--batch-size", str(BATCH_SIZE),
        "--n-workers", str(N_WORKERS), "--su2-bin", SU2_BIN, "--poll-seconds", "10",
        "--claim-ttl", str(CLAIM_TTL), "--explore-fraction", str(EXPLORE_FRACTION),
        # SU2 runs on LOCAL disk (fast — its live small-file I/O on the Drive mount was the earlier
        # low-utilization cause). The Drive LEDGER persists every design's parameters + result
        # (NaN for failures), which is all you need to see which failed AND 3D-render any design
        # (cells 11-12) — the disposable CFD mesh/scratch is not persisted, by design.
        "--cfd-out", "/content/cfd"]
if N_CYCLES is not None:   argv += ["--n-cycles", str(N_CYCLES)]
if INNER_ITER is not None: argv += ["--inner-iter", str(INNER_ITER)]
# Long-running + resumable: every eval is appended to the shared Drive ledger, so re-running
# this cell after a Colab drop resumes from the ledger (no work lost).
campaign.main(argv)
_dt_h = (time.time() - _t0) / 3600; _dn = len(read_ledger(SHARED_DIR)[0]) - _n0
print(f"\nthis session: {_dn} evals in {_dt_h:.2f} h"
      + (f"  ->  ~{_dt_h / max(_dn, 1) * N_WORKERS:.2f} h/eval wall" if _dn else ""))

## 7. Campaign summary — designs, averages, progression, best (safe anytime)

In [ ]:
from fanopt.bo.campaign_analysis import campaign_report, find_shards
# The pooled current campaign (seed = all prior deduped designs + new evals). To analyze ONLY the
# first (broken) run instead: shards = [s for v in find_shards(DRIVE_ROOT,"blade_campaign*").values() for s in v]
shards = [str(p) for p in SHARED_DIR.glob("evaluations_*.jsonl")]
r = campaign_report(shards)
print(f"unique designs: {r['unique_designs']}  (finite {r['finite']}, failed/NaN {r['failed_nan']})  sources={r['sources']}")
if r.get("j_fan"):
    p = r["progression"]
    print(f"J_fan  min/mean/max : {r['j_fan']['min']:+.2e} / {r['j_fan']['mean']:+.2e} / {r['j_fan']['max']:+.2e}")
    print(f"mass g min/mean/max : {r['mass_g']['min']:.0f} / {r['mass_g']['mean']:.0f} / {r['mass_g']['max']:.0f}")
    print("\n-- did it LEARN? (means, not the lucky max) --")
    print(f"  DoE(Sobol) mean {r['sobol_mean']:+.2e} (best {r['sobol_best']:+.2e})")
    print(f"  BO         mean {r['bo_mean']:+.2e} (best {r['bo_best']:+.2e})   "
          f"-> BO proposals are {'CONSISTENTLY better' if r['bo_mean']>r['sobol_mean'] else 'NOT better'} on average")
    qm = p["quartile_means"]
    if qm: print("  mean J_fan by quarter of the run: " + " -> ".join(f"{q:+.2e}" for q in qm) +
                 "  (rising = learning)")
    print("\n-- LEARNING DEPTH (fragmentation: each session only saw its OWN history) --")
    for s, st in sorted(p["by_session"].items()):
        fh, sh = st["first_half_mean"], st["second_half_mean"]
        lift = f"{(sh-fh)/abs(fh)*100:+.0f}%" if fh else "n/a"
        print(f"  {s}: {st['n']:3} designs | 1st-half mean {fh:+.2e} -> 2nd-half {sh:+.2e} ({lift}) | best {st['best']:+.2e}")
    print(f"  => deepest single-session learning = {max(st['n'] for st in p['by_session'].values())} designs "
          f"(NOT {r['finite']}); the sessions never pooled history.")
    print(f"\nPareto front: {r['pareto_count']} designs")
    print("top by J_fan (many are OVER the 100g cap):")
    for d in r["top_by_j_fan"][:5]:
        print(f"  J_fan={d['j_fan']:+.2e}  mass={d['mass_g']:.0f}g  {d['source']}  {d['design_hash']}")
    print("top UNDER the 100g mass cap (V1-eligible):")
    elig = r["top_under_mass_cap"]["100.0g"]
    for d in elig[:5] or [None]:
        print("  (none under 100g!)" if d is None else
              f"  J_fan={d['j_fan']:+.2e}  mass={d['mass_g']:.0f}g  {d['source']}  {d['design_hash']}")

## 8. Per-session J_fan progression (interactive — hover a dot for shape/J_fan/mass)

In [ ]:
import numpy as np
import plotly.graph_objects as go
from fanopt.bo.campaign_analysis import session_trajectories
# INTERACTIVE per-session progression — HOVER any dot for that design's shape / J_fan / mass.
traj = session_trajectories([str(pp) for pp in SHARED_DIR.glob("evaluations_*.jsonl")])
assert traj, "no designs yet — run the campaign (cell 6) first."
palette = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]
fig = go.Figure()
for k, (s, pts) in enumerate(sorted(traj.items())):
    col = palette[k % len(palette)]
    x = [d["eval"] for d in pts]; y = [d["j_fan"] for d in pts]
    hover = [(f"<b>{s}</b>  eval {d['eval']}<br>J_fan = {d['j_fan']:+.2e}<br>mass = {d['mass_g']:.0f} g"
              f"<br>shape = {d['peak']} / {d['interp']}<br>blades = {d['blade_count']}<br>{d['hash']}")
             for d in pts]
    fig.add_trace(go.Scatter(x=x, y=y, mode="markers", name=f"{s} ({len(pts)})",
                             marker=dict(size=6, color=col, opacity=0.45),
                             text=hover, hovertemplate="%{text}<extra></extra>"))
    w = max(5, len(y) // 15)
    rm = [float(np.mean(y[max(0, i - w + 1):i + 1])) for i in range(len(y))]
    fig.add_trace(go.Scatter(x=x, y=rm, mode="lines", name=f"{s} rolling mean",
                             line=dict(color=col, width=3), hoverinfo="skip"))
fig.update_layout(title="Per-session progression — HOVER a dot for design details (shape / J_fan / mass)",
                  xaxis_title="that session's own evaluation # (time order)", yaxis_title="J_fan",
                  height=560, legend=dict(orientation="h"))
fig.show()

## 9. DESIGN progression — how the wave SHAPE evolved over the run

In [ ]:
import matplotlib.pyplot as plt
from fanopt.bo.campaign_analysis import shape_evolution
# DESIGN progression: the rib-wave SHAPE of designs sampled across the run, dark(early)->bright(late).
ev = shape_evolution([str(p) for p in SHARED_DIR.glob("evaluations_*.jsonl")], n_samples=12)
assert ev, "no designs yet — run the campaign (cell 6) first."
cmap = plt.cm.viridis
plt.figure(figsize=(9, 5))
xk = range(1, len(ev[0]["knots_mm"]) + 1)
for d in ev:
    plt.plot(xk, d["knots_mm"], marker="o", color=cmap(d["eval_frac"]), alpha=0.85)
plt.xlabel("rib-bow knot  (1 = hub  ->  5 = tip)"); plt.ylabel("wave height (mm)")
plt.title("DESIGN progression — the rib-wave SHAPE over the run\n(dark = early designs -> bright = late designs)")
sm = plt.cm.ScalarMappable(cmap=cmap); sm.set_array([0, 1])
plt.colorbar(sm, ax=plt.gca(), label="run progress (early -> late)")
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
print("Each line is one design's wave hub->tip; watch the shape shift from early(dark) to late(bright).")
for d in ev:
    print(f"  eval {d['eval_index']:3d}  peak={d['peak']:3}  J={d['j_fan']:+.2e}  knots(mm)={d['knots_mm']}")

## 10. Surface-shape coverage — RIB-bow wave + PANEL aero grid (what shapes, is it skewed?)

In [ ]:
from fanopt.bo.campaign_analysis import panel_shape_summary, shape_summary
# WHAT SHAPES did we explore, and is the pool skewed to one type? (decodes the rib-bow wave)
sh = shape_summary([str(p) for p in SHARED_DIR.glob("evaluations_*.jsonl")])
assert sh["n"] > 0, "no decodable designs yet — run the campaign (cell 6) first."
print(f"designs decoded: {sh['n']}")
print(f"surface types (wave-peak/interp): {sh['type_counts']}")
print(f"wave-peak location counts       : {sh['peak_counts']}")
print("mean J_fan by peak location     : " + ", ".join(f"{k}={v:+.2e}" for k, v in sh['peak_mean_jfan'].items()))
print(f"blade_count counts              : {sh['blade_counts']}")
print(f"mean wave amplitude             : {sh['mean_amplitude_mm']:.1f} mm")
print(f"\nCONVERGENCE — did later designs drift to one type?")
print(f"  EARLY (first third) peaks: {sh['early_peak_dist']}")
print(f"  LATE  (last third)  peaks: {sh['late_peak_dist']}")
print("If LATE is dominated by one peak type, the search narrowed onto it — check whether that type "
      "actually has the best mean J_fan above, or if it just got stuck there (skew risk).")

# --- PANEL aero surface — the 24-of-33 codec dims the rib-bow summary above ignores ---------------
ps = panel_shape_summary([str(p) for p in SHARED_DIR.glob("evaluations_*.jsonl")])
print(f"\nPANEL aero surface (the free 4x3 grid — camber/zigzag/multi-hump lives here):")
print(f"  panel shape types           : {ps['class_counts']}")
print(f"  panel offset amplitude       : median {ps['panel_amp_mm']['median']:.2f} mm "
      f"(p10 {ps['panel_amp_mm']['p10']:.2f}, p90 {ps['panel_amp_mm']['p90']:.2f}, max {ps['panel_amp_mm']['max']:.2f})")
print(f"  rib-bow extent (for scale)   : median {ps['rib_bow_mm_median']:.1f} mm")
print(f"  PANEL AUTHORITY vs rib bow    : {ps['authority_pct_median']:.1f}%   "
      f"(<~5% => the rib meridian dominates J_fan; the panel is a maxed-but-tiny ripple)")
print(f"  panel knobs pinned to the containment limit: {ps['at_bound_pct_median']:.0f}% of 12 nodes "
      f"(high => the optimizer WANTED more panel travel than containment allowed => the lever is "
      f"rib-thickness/containment, not the codec)")

## 11. What the TOP designs look like (the winning wave shapes)

In [ ]:
from fanopt.bo.campaign_analysis import top_designs_shapes
# What do the TOP designs actually look like? RIB-bow wave (knots hub->tip, mm) AND the PANEL shape.
for d in top_designs_shapes([str(p) for p in SHARED_DIR.glob("evaluations_*.jsonl")], k=10):
    print(f"  J_fan={d['j_fan']:+.2e}  mass={d['mass_g']:3.0f}g  rib={d['peak']:3}/{d['interp']:6}  "
          f"knots(mm)={d['knots_mm']}  panel={d['panel_class']:6}({d['panel_amp_mm']:.2f}mm)  "
          f"blades={d['blade_count']}  {d['hash']}")

## 12. Which designs FAILED (from the Drive ledger)

In [ ]:
from fanopt.bo.campaign_analysis import failed_designs
# WHICH designs failed (NaN = infeasible geometry or a diverged CFD run) — read from the Drive ledger.
f = failed_designs([str(p) for p in SHARED_DIR.glob("evaluations_*.jsonl")])
print(f"{len(f)} failed designs:")
for d in f:
    print(f"  {d['hash']}  {d['source']}  peak={d['peak']}  blades={d['blade_count']}  knots(mm)={d['knots_mm']}")

## 13. Render the TOP 10 designs (3D inline + STEP export to Drive)

In [ ]:
import json
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import fanopt.geometry.blade_cad as blade_cad
from fanopt.geometry.blade_cad import blade_trimesh, export_blade_step
from fanopt.bo.blade_codec import decode
from fanopt.bo.campaign_analysis import classify_panel
blade_cad.N_RADIAL_SECTIONS = 40
# TOP 10 designs by J_fan, rebuilt from ledger PARAMETERS (no CFD needed). Plotly Mesh3d => DRAG to
# rotate, SCROLL to zoom. aspectmode="data" gives TRUE proportions: the rib bow is a gentle rise on a
# 165 mm blade, NOT the exaggerated wedge matplotlib's independently-auto-scaled axes drew.
rows = {}
for _f in SHARED_DIR.glob("evaluations_*.jsonl"):
    for _l in _f.read_text().splitlines():
        if _l.strip():
            _d = json.loads(_l)
            if isinstance(_d.get("j_fan"), (int, float)) and _d["j_fan"] == _d["j_fan"]:
                rows[_d["design_hash"]] = _d
top = sorted(rows.values(), key=lambda d: -d["j_fan"])[:10]
assert top, "no finite designs yet — run the campaign (cell 6) first."
out = SHARED_DIR / "step_exports"; out.mkdir(exist_ok=True)
meshes, titles = [], []
for k, d in enumerate(top):
    params = decode(np.array(d["vector"], dtype=float))
    v, faces = blade_trimesh(params, tol=0.001)
    meshes.append((v, faces))
    titles.append(f"#{k+1} J={d['j_fan']:+.1e}<br>{d['mass_kg']*1e3:.0f}g · panel:{classify_panel(params.panel_offsets_m)}")
    export_blade_step(params, str(out / f"top{k+1:02d}_{d['design_hash'][:8]}.step"))
fig = make_subplots(rows=2, cols=5, specs=[[{"type": "scene"}] * 5 for _ in range(2)],
                    subplot_titles=titles, horizontal_spacing=0.005, vertical_spacing=0.05)
for k, (v, faces) in enumerate(meshes):
    fig.add_trace(go.Mesh3d(x=v[:, 0], y=v[:, 1], z=v[:, 2], i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
                            intensity=v[:, 2], colorscale="Viridis", showscale=False),
                  row=k // 5 + 1, col=k % 5 + 1)
# TRUE aspect + hidden axes on every 3D scene — this is what removes the fake exaggerated angle.
fig.for_each_scene(lambda s: s.update(aspectmode="data", xaxis_visible=False,
                                      yaxis_visible=False, zaxis_visible=False))
fig.update_layout(height=680, margin=dict(l=0, r=0, t=70, b=0),
                  title_text="Top 10 by J_fan — DRAG to rotate, SCROLL to zoom (TRUE proportions)")
fig.show()
print(f"Also wrote 10 STEP files to {out} (Drive) — download for an interactive CAD viewer.")